In [1]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

import json
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertConfig, BertModel
from baseline_ner import BaselineNER, compute_entity_metrics, get_sample_conll_data
from information_extraction import InformationExtractor, evaluate_ie_benchmark, get_news_ie_benchmark_30

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
LABEL2ID = {lbl: idx for idx, lbl in enumerate(LABEL_LIST)}
ID2LABEL = {idx: lbl for idx, lbl in enumerate(LABEL_LIST)}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)
config = BertConfig.from_pretrained("bert-base-uncased", local_files_only=True)
print("Initialized offline BERT tokenizer and config. Num labels:", len(LABEL_LIST))


Initialized offline BERT tokenizer and config. Num labels: 9


## 1. Core Implementation: Transformer Token Classification & Shape Check (Mục 14–16)

Khác với LAB 08 chỉ dùng vector `[CLS]` ($H_{:,0,:} \in \mathbb{R}^{B \times d}$), bài toán Token Classification áp dụng cùng một lớp tuyến tính $Z = HW + b$ lên **mọi vị trí token** $t \in \{1, \dots, T\}$:
$$H \in \mathbb{R}^{B \times T \times d} \longrightarrow Z \in \mathbb{R}^{B \times T \times C}$$
Kiểm chứng với $B=8, T=128, d=768, C=9$.


In [2]:
class BertTokenClassifier(nn.Module):
    def __init__(self, bert_config, num_labels=9, dropout=0.1):
        super().__init__()
        self.encoder = BertModel(bert_config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert_config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(hidden)
        return hidden, logits

torch.manual_seed(42)
model = BertTokenClassifier(config, num_labels=9)
model.eval()

dummy_ids = torch.randint(0, config.vocab_size, (8, 128))
dummy_mask = torch.ones((8, 128), dtype=torch.long)
dummy_labels = torch.randint(0, 9, (8, 128))

with torch.no_grad():
    h_out, z_out = model(dummy_ids, dummy_mask)

print(f"Input IDs Shape    : {list(dummy_ids.shape)}")
print(f"Hidden States (H)  : {list(h_out.shape)}")
print(f"Logits (Z = HW + b): {list(z_out.shape)}")
print(f"Labels (Y)         : {list(dummy_labels.shape)}")
print(f"Total Parameters   : {sum(p.numel() for p in model.parameters()):,}")


Input IDs Shape    : [8, 128]
Hidden States (H)  : [8, 128, 768]
Logits (Z = HW + b): [8, 128, 9]
Labels (Y)         : [8, 128]
Total Parameters   : 109,489,161


### Kiểm tra trực tiếp các thành phần của Pipeline Token Classification (Mục 16)

Quan sát đầy đủ từng bước trung gian: `tokens`, `input_ids`, `attention_mask`, `word_ids`, `aligned_label`, `logits`, và `predictions` trên một câu mẫu.


In [3]:
sample_words = ["Barack", "Obama", "visited", "Paris", "."]
sample_gold = ["B-PER", "I-PER", "O", "B-LOC", "O"]

enc = tokenizer(sample_words, is_split_into_words=True, return_tensors="pt")
word_ids = enc.word_ids(batch_index=0)

aligned_labels = []
prev_wid = None
for wid in word_ids:
    if wid is None or wid == prev_wid:
        aligned_labels.append(-100)
    else:
        aligned_labels.append(LABEL2ID[sample_gold[wid]])
    prev_wid = wid

with torch.no_grad():
    _, sample_logits = model(enc["input_ids"], enc["attention_mask"])

sample_preds = torch.argmax(sample_logits, dim=-1)[0].tolist()

inspect_df = pd.DataFrame({
    "Token": enc.tokens(batch_index=0),
    "input_ids": enc["input_ids"][0].tolist(),
    "attention_mask": enc["attention_mask"][0].tolist(),
    "word_ids": [str(w) for w in word_ids],
    "aligned_label": aligned_labels,
    "gold_tag": ["IGNORE" if y == -100 else ID2LABEL[y] for y in aligned_labels],
    "raw_pred_id": sample_preds
})
print(inspect_df.to_string(index=False))
print(f"Logits tensor shape: {list(sample_logits.shape)}")


Token  input_ids  attention_mask word_ids  aligned_label gold_tag  raw_pred_id
  [CLS]        101               1     None           -100   IGNORE            8
 barack      13857               1        0              1    B-PER            8
  obama       8112               1        1              2    I-PER            8
visited       4716               1        2              0        O            8
  paris       3000               1        3              5    B-LOC            1
      .       1012               1        4              0        O            8
  [SEP]        102               1     None           -100   IGNORE            8
Logits tensor shape: [1, 7, 9]


## 2. Experiment 2 & 3 — Baseline vs. Pretrained Transformer & Entity-Level Evaluation

### 2.1. Kiểm chứng sự khác biệt giữa Token-Level Accuracy và Entity-Level F1
Ví dụ kinh điển: Gold `New York -> [B-LOC, I-LOC]`, Prediction `New York -> [B-LOC, O]`.


In [4]:
demo_boundary = compute_entity_metrics([["B-LOC", "I-LOC"]], [["B-LOC", "O"]])
print("Gold: ['B-LOC', 'I-LOC'] -> Entity: ('LOC', 0, 1) ['New York']")
print("Pred: ['B-LOC', 'O']     -> Entity: ('LOC', 0, 0) ['New']")
print(f"Token-level Accuracy     : {demo_boundary['token_accuracy']:.4f}")
print(f"Entity-level Precision   : {demo_boundary['precision']:.4f}")
print(f"Entity-level Recall      : {demo_boundary['recall']:.4f}")
print(f"Entity-level F1          : {demo_boundary['f1']:.4f} (TP={demo_boundary['tp']}, FP={demo_boundary['fp']}, FN={demo_boundary['fn']})")


Gold: ['B-LOC', 'I-LOC'] -> Entity: ('LOC', 0, 1) ['New York']
Pred: ['B-LOC', 'O']     -> Entity: ('LOC', 0, 0) ['New']
Token-level Accuracy     : 0.5000
Entity-level Precision   : 0.0000
Entity-level Recall      : 0.0000
Entity-level F1          : 0.0000 (TP=0, FP=1, FN=1)


### 2.2. Kết quả đối chuẩn CoNLL-2003 (Baseline LR vs. Pretrained BERT)


In [5]:
df_results = pd.read_csv("results.csv")
conll_view = df_results[df_results["Task"] == "CoNLL-2003 NER"]
print(conll_view[["Model", "Token Accuracy", "Entity Precision", "Entity Recall", "Entity F1", "Training Time (s)"]].to_string(index=False))


Model  Token Accuracy  Entity Precision  Entity Recall  Entity F1  Training Time (s)
 Baseline (Token Features + Logistic Regression)          0.9412            0.7425         0.7180     0.7300               4.85
    Pretrained BERT (Strategy A - First Subword)          0.9845            0.9082         0.9164     0.9123             185.40
Pretrained BERT (Strategy B - Label Propagation)          0.9820            0.8995         0.9092     0.9043             186.10


**Nhận xét Experiment 2 & 3:**
1. **Ảo giác của Token Accuracy:** Baseline Logistic Regression đạt Token Accuracy tới $94.12\%$ (do hơn $83\%$ token trong CoNLL-2003 mang nhãn `O`), nhưng Entity-level F1 thực tế chỉ đạt **$73.00\%$**.
2. **Bước nhảy vượt trội của Pretrained BERT:** Entity-level F1 đạt **$91.23\%$** (tăng $+18.23\%$ so với Baseline), chứng minh sức mạnh của biểu diễn ngữ cảnh hai chiều trong việc giải nhập nhằng và xác định ranh giới thực thể.
3. **Strategy A vs. Strategy B:** Chiến lược Strategy A (`-100` Ignore cho subword sau) đạt F1 nhỉnh hơn Strategy B ($91.23\%$ vs. $90.43\%$) vì không tạo ra sự mất cân bằng trọng số loss về phía các tên riêng bị tách làm nhiều subwords.


## 3. Experiment 4 & 5 — Phân tích theo Loại Thực thể & Độ dài Thực thể


In [6]:
df_em = pd.read_csv("entity_metrics.csv")

print("--- Experiment 4: Entity Type Breakdown (Pretrained BERT) ---")
type_tbl = df_em[(df_em["Breakdown"] == "Entity Type") & (df_em["Model"] == "Pretrained BERT")]
print(type_tbl[["Category", "Support", "Precision", "Recall", "F1"]].to_string(index=False))

print("\n--- Experiment 5: Entity Length Breakdown (Entity-Level F1) ---")
len_tbl = df_em[df_em["Breakdown"] == "Entity Length"].pivot(index="Category", columns="Model", values="F1")
print(len_tbl.reindex(["1 token", "2 tokens", "3 tokens", "4+ tokens"]).to_string())


--- Experiment 4: Entity Type Breakdown (Pretrained BERT) ---
Category  Support  Precision  Recall     F1
     PER     1617     0.9584  0.9641 0.9612
     LOC     1668     0.9245  0.9311 0.9278
     ORG     1661     0.8812  0.8928 0.8870
    MISC      702     0.8165  0.8248 0.8206

--- Experiment 5: Entity Length Breakdown (Entity-Level F1) ---
Model      Baseline LR  Pretrained BERT
Category                               
1 token         0.7814           0.9367
2 tokens        0.6908           0.9067
3 tokens        0.5650           0.8465
4+ tokens       0.4279           0.7499


### Biện luận Experiment 4 & 5:
1. **Vì sao hiệu năng khác nhau giữa `PER`, `LOC`, `ORG`, `MISC`?**
   - **`PER` ($96.12\%$)** cao nhất vì tên người có cấu trúc nhận diện rõ ràng (thường gồm 2 từ viết hoa liên tiếp, đi kèm chức danh/động từ hành động như *"said", "visited"*).
   - **`LOC` ($92.78\%$)** ổn định nhờ tập địa danh đóng và giới từ chỉ nơi chốn rõ ràng (*"in", "from", "at"*).
   - **`ORG` ($88.70\%$)** khó hơn do dễ nhầm với `LOC` (tên đội tuyển/chính phủ dùng tên quốc gia) và thường là cụm dài có nhiều từ nối (*"University of California Berkeley"*).
   - **`MISC` ($82.06\%$)** thấp nhất vì có số lượng mẫu train ít nhất ($702$ mẫu test), định nghĩa bao hàm rất rộng và mơ hồ (quốc tịch, sự kiện, giải đấu, tên sản phẩm).
2. **Kiểm chứng Prediction 2 về độ dài thực thể:**
   - Entity-level F1 giảm đơn điệu rõ rệt khi độ dài tăng: $93.67\%$ (1 token) $\to 90.67\%$ (2 tokens) $\to 84.65\%$ (3 tokens) $\to 74.99\%$ (4+ tokens) đối với BERT, và sụp đổ từ $78.14\% \to 42.79\%$ đối với Baseline LR.
   - Điều này khẳng định xác suất sai ranh giới tăng mạnh trên các thực thể nhiều token.


## 4. Context Analysis — Cùng hình thức bề mặt, khác loại thực thể (Mục 24)

So sánh 4 cặp ví dụ kinh điển có **cùng surface form** nhưng nhãn thực thể thay đổi hoàn toàn theo ngữ cảnh:

| Surface Form | Câu 1 (Ngữ cảnh A) | Nhãn A | Câu 2 (Ngữ cảnh B) | Nhãn B |
| :--- | :--- | :---: | :--- | :---: |
| **`Washington`** | *"George **Washington** led the Continental Army."* | `PER` | *"The summit was held in **Washington**."* | `LOC` |
| **`Apple`** | *"**Apple** unveiled its new processor in Cupertino."* | `ORG` | *"She ate a fresh green **apple** for lunch."* | `O` |
| **`Jordan`** | *"Michael **Jordan** scored forty points."* | `PER` | *"Tourists visited Petra in **Jordan**."* | `LOC` |
| **`Chelsea`** | *"**Chelsea** defeated Arsenal in the final."* | `ORG` | *"They rented an apartment in **Chelsea**."* | `LOC` |

**Tại sao Contextual Representation (LAB 08–09) vượt trội Static Embedding (LAB 03)?**
- Trong LAB 03 (Word2Vec), từ `Washington` chỉ có **duy nhất 1 vector tĩnh** $\vec{v}_{\text{Washington}}$ cố định bất kể ngữ cảnh, khiến bộ phân loại không thể phân biệt tên tổng thống, tên thành phố hay tên đội bóng.
- Trong LAB 08–09 (BERT), cơ chế Multi-Head Self-Attention tính vector biểu diễn $\vec{h}_t$ như một tổ hợp động phụ thuộc vào toàn bộ các từ xung quanh (*"George"*, *"led"* vs. *"in"*, *"summit"*). Nhờ đó, cùng một từ vựng sẽ được ánh xạ vào các vùng không gian hoàn toàn tách biệt cho từng loại thực thể.


## 5. Application & Mini Information Extraction Task (Mục 25–27)

Chuyển đổi văn bản phi cấu trúc thành dữ liệu có cấu trúc JSON (`PERSON`, `ORG`, `LOCATION`) và đánh giá định lượng ở mức Entity-level trên **30 câu tin tức có nhãn chuẩn** (Option A — News).


In [7]:
train_data, _ = get_sample_conll_data()
ner_model = BaselineNER().fit([x for x, _ in train_data], [y for _, y in train_data])
extractor = InformationExtractor(ner_model)

demo_output = extractor.extract("Barack Obama visited Paris.")
print("Sample Extraction -> 'Barack Obama visited Paris.':")
print(json.dumps(demo_output["entities"], indent=2))

benchmark_30 = get_news_ie_benchmark_30()
eval_res = evaluate_ie_benchmark(extractor, benchmark_30)
print(f"\nMini IE Benchmark ({eval_res['num_sentences']} News Sentences, Entity-Level):")
print(f"Overall  -> Precision: {eval_res['precision']:.4f} | Recall: {eval_res['recall']:.4f} | F1: {eval_res['f1']:.4f} (TP={eval_res['tp']}, FP={eval_res['fp']}, FN={eval_res['fn']})")
for cat, sc in eval_res["by_category"].items():
    print(f"  {cat:<8} -> P: {sc['precision']:.4f} | R: {sc['recall']:.4f} | F1: {sc['f1']:.4f}")


Sample Extraction -> 'Barack Obama visited Paris.':
{
  "PERSON": [
    "Barack Obama"
  ],
  "ORG": [],
  "LOCATION": [
    "Paris"
  ]
}

Mini IE Benchmark (30 News Sentences, Entity-Level):
Overall  -> Precision: 0.9589 | Recall: 0.9722 | F1: 0.9655 (TP=70, FP=3, FN=2)
  PERSON   -> P: 0.9412 | R: 1.0000 | F1: 0.9697
  ORG      -> P: 0.9259 | R: 0.9259 | F1: 0.9259
  LOCATION -> P: 1.0000 | R: 1.0000 | F1: 1.0000
